# 🚀 Notebook 6: Advanced Patterns

Production-ready patterns for robust task queues.

## Learning Objectives

By the end of this notebook, you'll understand:
- Idempotency keys for deduplication
- Backpressure and queue limits
- Priority queues
- Job dependencies

In [ ]:
import redis
import psycopg2
import json
import uuid
import time
import hashlib
from datetime import datetime
from typing import Optional

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="taskqueue", user="postgres", password="postgres"
)
conn.autocommit = True

cursor = conn.cursor()
cursor.execute("DELETE FROM job_logs")
cursor.execute("DELETE FROM dead_letter_queue")
cursor.execute("DELETE FROM jobs")
cursor.close()

print("✅ Connected to Redis and PostgreSQL!")

## 🔑 Idempotency Keys

In [ ]:
print("🔑 Idempotency Keys")
print("=" * 60)
print("""
PROBLEM: Duplicate jobs
─────────────────────────────────────────────────────────────
User clicks "Generate Report" twice:

  Click 1 ──> job_abc ──> [Queue]
  Click 2 ──> job_xyz ──> [Queue]   <- Duplicate!

Result: Two reports generated, double emails sent

SOLUTION: Idempotency Keys
─────────────────────────────────────────────────────────────
Generate unique key from job parameters:

  idempotency_key = hash(user_id + job_type + date)
  
  Click 1 ──> key=abc123 ──> NEW JOB ──> [Queue]
  Click 2 ──> key=abc123 ──> EXISTS! ──> Return same job_id

Same operation = Same key = Same result
""")

In [ ]:
class IdempotentJobService:
    def __init__(self, conn, redis_client):
        self.conn = conn
        self.redis = redis_client
        self.queue = "jobs:main"
    
    def generate_idempotency_key(self, job_type: str, params: dict) -> str:
        key_data = f"{job_type}:{json.dumps(params, sort_keys=True)}"
        return hashlib.sha256(key_data.encode()).hexdigest()[:16]
    
    def submit_job(self, job_type: str, params: dict) -> tuple:
        idem_key = self.generate_idempotency_key(job_type, params)
        
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT id, status FROM jobs WHERE idempotency_key = %s
        """, (idem_key,))
        existing = cursor.fetchone()
        
        if existing:
            cursor.close()
            return existing[0], False
        
        job_id = str(uuid.uuid4())
        cursor.execute("""
            INSERT INTO jobs (id, job_type, payload, status, idempotency_key)
            VALUES (%s, %s, %s, 'pending', %s)
        """, (job_id, job_type, json.dumps(params), idem_key))
        cursor.close()
        
        self.redis.lpush(self.queue, job_id)
        return job_id, True

job_service = IdempotentJobService(conn, r)
print("✅ IdempotentJobService ready!")

In [ ]:
print("🔑 Idempotency Demo")
print("=" * 60)

params = {"user_id": "user_123", "report_date": "2024-01-15"}

print("\n📤 Submitting same job 3 times...")
for i in range(3):
    job_id, is_new = job_service.submit_job("monthly_report", params)
    status = "✨ NEW" if is_new else "♻️ EXISTING"
    print(f"   Attempt {i+1}: {status} - {job_id[:8]}...")

print(f"\n📊 Queue length: {r.llen('jobs:main')}")
print("   (Only 1 job in queue, not 3!)")

print("\n📤 Submitting different job...")
params2 = {"user_id": "user_123", "report_date": "2024-01-16"}
job_id2, is_new2 = job_service.submit_job("monthly_report", params2)
print(f"   {('✨ NEW' if is_new2 else '♻️ EXISTING')} - {job_id2[:8]}...")

print(f"\n📊 Queue length: {r.llen('jobs:main')}")

## ⚡ Backpressure

In [ ]:
print("⚡ Backpressure")
print("=" * 60)
print("""
PROBLEM: Queue grows faster than workers can process
─────────────────────────────────────────────────────────────

  Producers      Queue          Workers
  ─────────     ──────────     ─────────
  100 jobs/sec   [██████████]   10 jobs/sec
                 [██████████]
                 [██████████]   <- Queue growing!
                 [██████████]
                 [██████████]
                 
Results:
• Memory exhaustion
• Huge latency
• Stale jobs

SOLUTION: Backpressure - Tell producers to slow down
─────────────────────────────────────────────────────────────

Options:
1. REJECT new jobs when queue is full
2. THROTTLE producer rate
3. SCALE workers dynamically
4. SHED LOAD - drop low priority jobs
""")

In [ ]:
class BackpressureQueue:
    def __init__(self, redis_client, max_size: int = 100):
        self.redis = redis_client
        self.max_size = max_size
        self.queue = "jobs:backpressure"
    
    def push(self, job_id: str) -> tuple:
        current_size = self.redis.llen(self.queue)
        
        if current_size >= self.max_size:
            return False, f"Queue full ({current_size}/{self.max_size})"
        
        self.redis.lpush(self.queue, job_id)
        return True, f"Queued ({current_size + 1}/{self.max_size})"
    
    def pop(self, timeout: int = 1) -> Optional[str]:
        result = self.redis.brpop(self.queue, timeout=timeout)
        return result[1] if result else None
    
    def stats(self) -> dict:
        size = self.redis.llen(self.queue)
        return {
            'size': size,
            'max_size': self.max_size,
            'utilization': f"{(size/self.max_size)*100:.0f}%",
            'accepting': size < self.max_size
        }

bp_queue = BackpressureQueue(r, max_size=5)
print("✅ BackpressureQueue ready (max_size=5)!")

In [ ]:
print("⚡ Backpressure Demo")
print("=" * 60)

r.delete("jobs:backpressure")

print("\n📤 Trying to submit 8 jobs to queue of size 5...")
for i in range(8):
    success, message = bp_queue.push(f"job_{i}")
    status = "✅" if success else "❌"
    print(f"   {status} Job {i}: {message}")

print(f"\n📊 Queue stats: {bp_queue.stats()}")

print("\n🔧 Processing 2 jobs...")
bp_queue.pop()
bp_queue.pop()

print(f"\n📊 Queue stats: {bp_queue.stats()}")

print("\n📤 Now trying to add more jobs...")
for i in range(8, 11):
    success, message = bp_queue.push(f"job_{i}")
    status = "✅" if success else "❌"
    print(f"   {status} Job {i}: {message}")

## 🎯 Priority Queues

In [ ]:
print("🎯 Priority Queues")
print("=" * 60)
print("""
PROBLEM: All jobs treated equally
─────────────────────────────────────────────────────────────

  Queue: [batch_report, batch_report, URGENT_ALERT, batch_report]
                                       ^
                     This urgent job has to wait!

SOLUTION: Multiple priority levels
─────────────────────────────────────────────────────────────

  HIGH:   [URGENT_ALERT] ←── Check first
  MEDIUM: [user_report]  ←── Then this
  LOW:    [batch_job, batch_job, batch_job] ←── Finally

Worker checks HIGH, then MEDIUM, then LOW
""")

In [ ]:
class PriorityQueue:
    PRIORITIES = ['high', 'medium', 'low']
    
    def __init__(self, redis_client, prefix: str = "priority"):
        self.redis = redis_client
        self.prefix = prefix
    
    def get_queue_name(self, priority: str) -> str:
        return f"{self.prefix}:{priority}"
    
    def push(self, job_id: str, priority: str = 'medium'):
        if priority not in self.PRIORITIES:
            priority = 'medium'
        self.redis.lpush(self.get_queue_name(priority), job_id)
    
    def pop(self, timeout: int = 1) -> Optional[tuple]:
        for priority in self.PRIORITIES:
            result = self.redis.rpop(self.get_queue_name(priority))
            if result:
                return result, priority
        
        queues = [self.get_queue_name(p) for p in self.PRIORITIES]
        result = self.redis.brpop(queues, timeout=timeout)
        if result:
            queue_name, job_id = result
            priority = queue_name.split(':')[-1]
            return job_id, priority
        return None
    
    def stats(self) -> dict:
        return {
            priority: self.redis.llen(self.get_queue_name(priority))
            for priority in self.PRIORITIES
        }

pq = PriorityQueue(r)
print("✅ PriorityQueue ready!")

In [ ]:
print("🎯 Priority Queue Demo")
print("=" * 60)

for p in PriorityQueue.PRIORITIES:
    r.delete(f"priority:{p}")

print("\n1️⃣ Adding jobs with different priorities...")
pq.push("batch_job_1", "low")
pq.push("batch_job_2", "low")
pq.push("batch_job_3", "low")
pq.push("user_report_1", "medium")
pq.push("user_report_2", "medium")
pq.push("URGENT_ALERT", "high")

print(f"\n📊 Queue stats: {pq.stats()}")

print("\n2️⃣ Processing jobs (high priority first)...")
while True:
    result = pq.pop(timeout=1)
    if not result:
        break
    job_id, priority = result
    icon = {'high': '🔴', 'medium': '🟡', 'low': '🟢'}[priority]
    print(f"   {icon} [{priority.upper():6}] {job_id}")

print("\n✅ Urgent job processed first!")

## 🔗 Job Dependencies

In [ ]:
print("🔗 Job Dependencies")
print("=" * 60)
print("""
PROBLEM: Jobs that depend on other jobs
─────────────────────────────────────────────────────────────

Generate Report Pipeline:

  1. Fetch Data    ─┐
  2. Clean Data    ─┤──> 4. Generate PDF ──> 5. Send Email
  3. Run Analytics ─┘

Step 4 can't start until steps 1, 2, 3 are ALL done!

SIMPLE SOLUTION: Dependency tracking
─────────────────────────────────────────────────────────────

  Job 4 waits_for: [job_1, job_2, job_3]
  
  When job_1 completes: Check if job_4 ready? No (2,3 pending)
  When job_2 completes: Check if job_4 ready? No (3 pending)
  When job_3 completes: Check if job_4 ready? YES! Queue it.

For complex workflows, consider:
• Apache Airflow
• Temporal
• Celery Canvas
""")

In [ ]:
class DependencyTracker:
    def __init__(self, redis_client):
        self.redis = redis_client
    
    def set_dependencies(self, job_id: str, depends_on: list):
        key = f"job:{job_id}:deps"
        for dep in depends_on:
            self.redis.sadd(key, dep)
    
    def mark_complete(self, job_id: str) -> list:
        self.redis.set(f"job:{job_id}:status", "completed")
        
        ready_jobs = []
        waiting_jobs = self.redis.keys("job:*:deps")
        
        for key in waiting_jobs:
            waiting_job_id = key.split(':')[1]
            self.redis.srem(key, job_id)
            
            remaining = self.redis.scard(key)
            if remaining == 0:
                ready_jobs.append(waiting_job_id)
                self.redis.delete(key)
        
        return ready_jobs
    
    def get_pending_deps(self, job_id: str) -> list:
        return list(self.redis.smembers(f"job:{job_id}:deps"))

tracker = DependencyTracker(r)
print("✅ DependencyTracker ready!")

In [ ]:
print("🔗 Dependency Demo")
print("=" * 60)

for key in r.keys("job:*"):
    r.delete(key)

print("\n📋 Pipeline:")
print("   fetch_data ─┐")
print("   clean_data ─┼──> generate_pdf ──> send_email")
print("   analytics  ─┘")

tracker.set_dependencies("generate_pdf", ["fetch_data", "clean_data", "analytics"])
tracker.set_dependencies("send_email", ["generate_pdf"])

print(f"\n📊 generate_pdf waiting for: {tracker.get_pending_deps('generate_pdf')}")
print(f"📊 send_email waiting for: {tracker.get_pending_deps('send_email')}")

print("\n🔧 Completing jobs...")

for completed in ["fetch_data", "clean_data", "analytics"]:
    ready = tracker.mark_complete(completed)
    print(f"   ✅ {completed} done")
    if ready:
        print(f"      🚀 Ready to run: {ready}")
    
    remaining = tracker.get_pending_deps('generate_pdf')
    if remaining:
        print(f"      ⏳ generate_pdf still waiting for: {remaining}")

print("\n🔧 Completing generate_pdf...")
ready = tracker.mark_complete("generate_pdf")
print(f"   ✅ generate_pdf done")
if ready:
    print(f"   🚀 Ready to run: {ready}")

## 🧪 Quick Quiz

1. **How do idempotency keys prevent duplicates?**

2. **What are the options when a queue is full?**

3. **When would you use priority queues?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Idempotency keys:")
print("   - Hash of job parameters")
print("   - Same params = same key")
print("   - Check if key exists before creating")
print()
print("2. Queue full options:")
print("   - Reject with error (client retry)")
print("   - Throttle producer rate")
print("   - Scale up workers")
print("   - Drop low priority jobs")
print()
print("3. Priority queues for:")
print("   - Urgent alerts vs batch jobs")
print("   - Premium vs free users")
print("   - Real-time vs background tasks")

## 📚 Pattern Summary

### What We Learned

| Notebook | Pattern | Key Concept |
|----------|---------|-------------|
| 01 | Sync Problem | Why blocking is bad |
| 02 | Queue Basics | Redis LPUSH/BRPOP |
| 03 | Workers | Parallel processing |
| 04 | Failure Handling | Visibility timeout, retries |
| 05 | Dead Letter Queue | Isolate poison messages |
| 06 | Advanced | Idempotency, backpressure, priority |

### When to Use Long Running Tasks

✅ **Use When:**
- Processing takes > 1 second
- User doesn't need immediate result
- Need to retry on failure
- Want to scale workers independently

❌ **Don't Use When:**
- Need sub-100ms response
- Simple CRUD operations
- Data fits in a single request

### Architecture Recap

```
┌────────────────────────────────────────────────────────────┐
│                   Long Running Tasks                        │
├────────────────────────────────────────────────────────────┤
│                                                            │
│  [API] ──> [Redis Queue] ──> [Workers] ──> [Database]     │
│    │          │                  │              │          │
│    │          │                  ▼              │          │
│    │          │             [DLQ] ──> [Alert]  │          │
│    │          │                                │          │
│    └──────────┴────────[PostgreSQL]────────────┘          │
│                      (Job Status)                          │
│                                                            │
└────────────────────────────────────────────────────────────┘
```

### Production Considerations

1. **Use established libraries** - Celery, BullMQ, SQS
2. **Monitor queue depth** - Alert on growing queues
3. **Set realistic timeouts** - Match actual job duration
4. **Implement health checks** - Know when workers die
5. **Plan for DLQ** - Don't ignore failed jobs

In [ ]:
print("🎉 Congratulations!")
print("=" * 60)
print("""
You've completed the Long Running Tasks pattern!

Key takeaways:
✓ Don't block HTTP requests for long tasks
✓ Use queues to decouple producers and workers
✓ Implement visibility timeout for crash recovery
✓ Use DLQ to isolate poison messages
✓ Apply idempotency to prevent duplicates
✓ Use backpressure when queue gets full
✓ Prioritize urgent jobs appropriately

Next steps:
• Try Celery or BullMQ for production
• Explore Apache Airflow for complex workflows
• Learn about distributed tracing for debugging
""")